This notebook contains code for prompting LLMs to translate NL to structured NL or TL

Currently, the notebook is set up to generate FRETish, and you can all cells to test it
You can configure to select the benchmark, and target structured NL (fretish or PSP)

In the cell below, you must provide your GEMINI or OPENAI API KEY

To convenientialy run all cells: Select Run -> Run all cells option from the top of this notebook to view the plots

In [1]:
import os
os.environ["STRUCTNL_MODE"] = "fretish" #or "PSP"
#No config needed, since we're using Ollama
#os.environ["GEMINI_API_KEY"] ="TODO"
#os.environ["OPENAI_API_KEY"] ="TODO"
# Optionally override Ollama URL (default: http://localhost:11434/v1)
#os.environ["OLLAMA_BASE_URL"] = "http://localhost:11434/v1"

In [2]:
import pandas as pd
import json
import itertools
os.environ["DATA_HOME_DIR"] = "./metadata"
#os.environ["GOOGLE_APPLICATION_CREDENTIALS"] =""

In [3]:
import data_loader
import llm_prompt
import nl2structnl
import nl2ltl
import nl2spec
import NL2TL
import deepstl
import synthTL

['timing options', 'timing bool exps', 'scope 1 options', 'scope 1 bool exps', 'scope 2 options', 'scope 2 bool exps']


In [4]:
if os.getenv("STRUCTNL_MODE") == "fretish":
    data_home_dir = "./benchmarks/fret_specs/"
else:
    data_home_dir = "./benchmarks/psp_specs/"


In [5]:
#select the list of benchmarks

#dataset_list = ["Ventilator","FSM-AP","FSM-S","REG","RobotExplain","deepstl-test"]
#dataset_list = ["Thales"]
dataset_list = ["FSM-S"]

In [6]:
#select model and number of candidates to generate per requirement

num_trial = 1
model_list = ["qwen2.5:32b"]
#model_list = ["gemini-2.5-flash"]
#model_list = ["gpt-4.1"]

In [7]:
#specify directory to save results

save_dir = "./ollama_outputs"

In [8]:
#specify prompting approach, currently configured to use ARTEMIS "nl2structnl"

all_methods = ["nl2structnl","nl2ltl","nl2spec","NL2TL","NL2TL-FT","deepstl","nl2ltltemplate","synthtl"]
cur_methods = ["nl2structnl"]
cur_methods

['nl2structnl']

In [9]:
def is_need_run(save_dir,cur_dataset_name,row_idx,model,num_trial,cur_method):
    cur_exp_name = f"{save_dir}/{cur_dataset_name}-{row_idx}_model-{model}_trials-{num_trial}"
    if not os.path.exists(cur_exp_name + "_" + cur_method + ".json"):
        return True
    cur_outputs, output_ltl_list = data_loader.load_outputs(save_dir,cur_dataset_name,row_idx,model,num_trial,cur_method,cur_mode=None,structnl=os.getenv("STRUCTNL_MODE"))
    return len(output_ltl_list) == 0

In [10]:
def get_resume_prev_outputs(save_dir,cur_dataset_name,row_idx,model,num_trial,cur_method):
    prev_outputs = []
    for cur_num_trial in range(1,num_trial+1):
        cur_exp_name = f"{save_dir}/{cur_dataset_name}-{row_idx}_model-{model}_trials-{cur_num_trial}"
        if os.path.exists(cur_exp_name + "_" + cur_method + ".json"):
            cur_outputs, output_ltl_list = data_loader.load_outputs(save_dir,cur_dataset_name,row_idx,model,cur_num_trial,cur_method,cur_mode=None,structnl=os.getenv("STRUCTNL_MODE"))
            prev_outputs = cur_outputs
    print("found prev outputs:",len(prev_outputs))
    return prev_outputs

In [11]:
import time
def run_single_task(cur_dataset_name,model,row_idx, cur_method):
    cur_df_file = data_home_dir + cur_dataset_name + "/PlausibleSpecs.xlsx"
    df = pd.read_excel(cur_df_file, engine='openpyxl')
    cur_exp_name = f"{save_dir}/{cur_dataset_name}-{row_idx}_model-{model}_trials-{num_trial}"
    #if row_idx < len(df) and not os.path.exists(cur_exp_name + "_" + cur_method + ".json"):
    if row_idx < len(df) :#and is_need_run(save_dir,cur_dataset_name,row_idx,model,num_trial,cur_method):
        print("row_idx:", cur_dataset_name,row_idx,model,cur_method)
        is_done = False
        while not is_done:
            #try:
            prev_outputs = []
            input_nl = df.iloc[row_idx]["NL"]
            print("input:")
            print(input_nl)
            ap_dict = data_loader.load_vars(data_home_dir,cur_dataset_name,row_idx=row_idx)
            if cur_method == "nl2structnl":
                outputs = llm_prompt.get_formalizations_loop(input_nl, ap_dict, nl2structnl.get_nl2structnl_translation, num_trial=num_trial, model=model,prev_outputs=prev_outputs)
            elif cur_method == "nl2structnl-reflect":
                prev_outputs = get_resume_prev_outputs(save_dir,cur_dataset_name,row_idx,model,num_trial,cur_method)
                outputs = llm_prompt.get_formalizations_loop(input_nl, ap_dict, nl2structnl.get_nl2structnl_translation, num_trial=num_trial, model=model,prev_outputs=prev_outputs,mode="reflect")
            elif cur_method == "nl2structnl_dcmp":
                dcmp, dcmp_list = nl2structnl.get_dcmp_map_from_row(df.iloc[row_idx])
                outputs = llm_prompt.get_formalizations_loop(input_nl, ap_dict, nl2structnl.get_nl2structnl_translation, num_trial=num_trial, model=model, dcmp=dcmp,prev_outputs=prev_outputs)
            elif cur_method == "nl2ltltemplate":
                outputs = llm_prompt.get_formalizations_loop(input_nl, ap_dict, nl2ltl.get_nl2ltltemplate_translation, num_trial, model=model,prev_outputs=prev_outputs)
            elif cur_method == "nl2ltl":
                outputs = llm_prompt.get_formalizations_loop(input_nl, ap_dict, nl2ltl.get_nl2ltl_translation, num_trial, model=model,prev_outputs=prev_outputs)
            elif cur_method == "nl2spec":
                outputs = llm_prompt.get_formalizations_loop(input_nl, ap_dict, nl2spec.get_nl2spec_translation, num_trial, model=model,prev_outputs=prev_outputs)
            elif cur_method == "NL2TL":
                outputs = llm_prompt.get_formalizations_loop(input_nl, ap_dict, NL2TL.get_NL2TL_translation, num_trial, model=model,prev_outputs=prev_outputs)
            elif cur_method == "NL2TL-FT":
                outputs = llm_prompt.get_formalizations_loop(input_nl, ap_dict, NL2TL.get_NL2TL_translation, num_trial, model=model,mode="FT",prev_outputs=prev_outputs)
            elif cur_method == "deepstl":
                outputs = llm_prompt.get_formalizations_loop(input_nl, ap_dict, deepstl.get_nl2ltl_translation, num_trial, model=model,prev_outputs=prev_outputs)
            elif cur_method == "synthtl":
                outputs = llm_prompt.get_formalizations_loop(input_nl, ap_dict, synthTL.get_synthtl_translation, num_trial, model=model,prev_outputs=prev_outputs)        
            else:
                raise ValueError(f"Unknown method: {cur_method}")
            #for output in outputs:
            #    print(output['output_structured_natural_language'])
            cur_exp_name = f"{save_dir}/{cur_dataset_name}-{row_idx}_model-{model}_trials-{num_trial}"
            with open(cur_exp_name + "_" + cur_method + ".json", "w") as json_file:
                json.dump(outputs, json_file)
            print(row_idx,cur_method,"done!","num outputs:",len(outputs))
            is_done = True
            #except Exception as e:
            #    print(f"Error during generation: {e}")
            #    time.sleep(30)
        return outputs

In [12]:
row_idx_range = None

tasks = []
for cur_dataset_name in dataset_list:
    for model in model_list:
        for method in cur_methods:
            cur_df_file = data_home_dir + cur_dataset_name + "/PlausibleSpecs.xlsx"
            df = pd.read_excel(cur_df_file, engine='openpyxl')
            for row_idx in range(len(df)):
                if row_idx_range is None or row_idx in row_idx_range:
                    tasks.append((cur_dataset_name,model,row_idx,method))

In [13]:
#run the LLM on each requirement in the benchmarks
for cur_dataset_name,model,row_idx, cur_method in tasks:
    outputs = run_single_task(cur_dataset_name,model,row_idx, cur_method)
    print("output:")
    print(outputs)
    #remove break after testing a single run, to process entire dataset
    #after generating outputs, then run compute_accuracy_metrics.ipynb
    #then do Plot_results.ipynb LATER when you have all versions good
    break

row_idx: FSM-S 0 qwen2.5:32b nl2structnl
input:
The sensor shall change states from NOMINAL to FAULT when limits are exceeded.
error msg: invalid output format: 1 validation error for StructuredNLTranslations
translations.0.decision3
  Input should be 'immediately satisfy bool_exp3', 'within N_DURATION ticks satisfy bool_exp3', 'after N_DURATION ticks satisfy bool_exp3', 'until bool_exp4, satisfy bool_exp3', 'always satisfy bool_exp3', 'never satisfy bool_exp3', 'at the next timepoint satisfy bool_exp3', 'eventually satisfy bool_exp3', 'for N_DURATION ticks satisfy bool_exp3' or 'before bool_exp4, satisfy bool_exp3' [type=literal_error, input_value='upon bool_exp2, satisfy bool_exp3', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
error msg: invalid output format: 1 validation error for StructuredNLTranslations
translations.0.decision3
  Input should be 'immediately satisfy bool_exp3', 'within N_DURATION ticks satisfy bool_exp3', 'aft

APIConnectionError: Connection error.